In [3]:
from langchain_groq import ChatGroq

from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

response = llm.invoke("What is the best phone under 15000?")
print(response.content)

/media/sathish/New Volume/.Trash-1000/files/AI-LLM-OPS.3/AI-LLM-OPS/flipkart-recommnedation-system/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The best phone under 15000 can vary depending on personal preferences, needs, and the latest market trends. However, here are some top options to consider:

1. **Redmi 12**: This phone offers a 6.7-inch display, 50MP primary camera, 5000mAh battery, and up to 8GB of RAM. It's available for around 13,000.
2. **Samsung Galaxy M33**: This phone features a 6.6-inch display, 50MP primary camera, 5000mAh battery, and up to 8GB of RAM. It's priced around 14,000.
3. **Realme 9 Pro**: This phone boasts a 6.4-inch display, 48MP primary camera, 5000mAh battery, and up to 8GB of RAM. It's available for around 13,000.
4. **Xiaomi Poco X4 Pro**: This phone offers a 6.67-inch display, 48MP primary camera, 5000mAh battery, and up to 8GB of RAM. It's priced around 14,000.
5. **Google Pixel 6a**: This phone features a 6.1-inch display, 12.2MP primary camera, 4410mAh battery, and up to 6GB of RAM. It's available for around 14,000.

When choosing the best phone under 15000, consider the following factors:

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "You are a expert in eletronics. Answer clearly .\n Question: {question}"
)

chain = prompt | llm

response = chain.invoke({"question": "What is the best phone under 15000?"})
print(response.content)

As an expert in electronics, I can provide you with some of the best phone options available in the market under 15000. Please note that the prices may vary depending on the region and availability.

Based on recent market trends and customer reviews, here are some of the top picks:

1. **Xiaomi Redmi 10**: This phone offers a 6.5-inch HD+ display, a powerful MediaTek Helio G88 processor, up to 6GB of RAM, and a 5000mAh battery. It also features a quad-camera setup with a 50MP primary sensor. Price: around 12,000 - 14,000 INR.
2. **Samsung Galaxy M13**: This phone features a 6.5-inch HD+ display, a large 5000mAh battery, and a quad-camera setup with a 50MP primary sensor. It's powered by a MediaTek Helio G85 processor and comes with up to 4GB of RAM. Price: around 10,000 - 13,000 INR.
3. **Realme C35**: This phone offers a 6.5-inch HD+ display, a powerful MediaTek Helio G35 processor, up to 4GB of RAM, and a 5000mAh battery. It also features a triple-camera setup with a 50MP primary se

In [5]:
import pandas as pd

df = pd.read_csv("../data/flipkart_product_review.csv")
df.head()

,product_id,product_title,rating,summary,review
0,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,1-more flexible2-bass is very high3-sound clar...
1,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,Super sound and good looking I like that prize
2,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Super!,Very much satisfied with the device at this pr...
3,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Super!,"Nice headphone, bass was very good and sound i..."
4,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,Sound quality super battery backup super quali...


In [6]:
from langchain_core.documents import Document

documents  = []


documents =[ Document(
        page_content= f""" Product : {row['product_title']} \n Review : {row['review']} \n Rating : {row['rating']} """,
        metadata={"product_id": row['product_id']}) 
    for _, row in df.iterrows() ]

len(documents)

450

In [17]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(documents, embedding)

/tmp/ipykernel_1905206/2336111496.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6222.02it/s]


In [18]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

docs = retriever.invoke("best headphone with good battery")

for d in docs:
    print(d.page_content[:200])
    print("----")

 Product : BoAt Airdopes 131 Bluetooth Headset 
 Review : Good for the priceSound if loud and clear. Bass is there but not best as these are not in-ear type. Call quality is good. And battery lasts 3.
----
 Product : OnePlus Bullets Wireless Z Bass Edition Bluetooth Headset 
 Review : Good battery back up, sound should be improved 
 Rating : 4 
----
 Product : OnePlus Bullets Wireless Z Bluetooth Headset 
 Review : first of all with mi 18 watt charger,it got full charged in 20mins.that was amazing and 16-18hrs playtime is best at this price range
----


In [19]:
context = "\n\n".join([d.page_content for d in docs])

prompt = f"""
Answer based on context only.

Context:
{context}

Question:
Best headphone with battery life?
"""

response = llm.invoke(prompt)
print(response.content)

OnePlus Bullets Wireless Z Bluetooth Headset. It has a battery life of 16-18 hours.


In [20]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | ChatPromptTemplate.from_template(
        "Answer using only this context:\n{context}\n\nQuestion: {question}"
    )
    | llm
    | StrOutputParser()
)

In [21]:
rag_chain.invoke("best headphones under 15000")

'Based on the given reviews, here are some recommendations for the best headphones under 15000:\n\n1. OnePlus Bullets Wireless Z Bluetooth Headset: This product has received a 5-star rating and is considered the "BEST under 2k". It has a magnetic control, comfortable neck band, and great battery life.\n\n2. realme Buds 2 Wired Headset: Although it\'s a wired headset, it has received a 5-star rating and is considered the best pair of earphones under Rs. 600. It\'s lightweight, well-fitted, and has a long-lasting wire.\n\nHowever, since the question is about headphones under 15000, the realme Buds 2 might not fit the budget. \n\nSo, the best option under 15000 would be the OnePlus Bullets Wireless Z Bluetooth Headset.'

In [23]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [ ]:

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


In [26]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
rewrite_prompt = ChatPromptTemplate.from_messages([
            ("system", "Rewrite the user query into a standalone question."),
            MessagesPlaceholder("chat_history"),
            ("human", "{input}")
        ])


In [27]:
rewrite_chain = (
            rewrite_prompt
            | llm
            | StrOutputParser()
        )


In [28]:
retrieval_chain = (
            RunnableLambda(lambda x: {
                "input": x["input"],
                "chat_history": x.get("chat_history", [])
            })
            | rewrite_chain
            | retriever
        )

In [29]:
qa_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "You're an e-commerce bot answering product-related queries.\n"
             "Use ONLY the context below.\n\nCONTEXT:\n{context}"
            ),
            MessagesPlaceholder("chat_history"),
            ("human", "{input}")
        ])

In [32]:
rag_chain = (
            {
                "context": retrieval_chain | RunnableLambda(format_docs),

                # Pass input correctly
                "input": RunnableLambda(lambda x: x["input"]),

                # Pass history correctly
                "chat_history": RunnableLambda(lambda x: x.get("chat_history", []))
            }
            | qa_prompt
            | llm
            | StrOutputParser()
        )


In [33]:
run = RunnableWithMessageHistory(
            rag_chain,
            get_session_history,
            input_messages_key="input",
            history_messages_key="chat_history",
        )

In [35]:


response = run.invoke(
    {"input": "best headphones under 15000"},
    config={"configurable": {"session_id": "user1"}}
)

print(response)

Based on the reviews provided, here are some suggestions for the best headphones under 15000:

1. **BoAt Airdopes 131 Bluetooth Headset**: This is a good option for those who want a budget-friendly Bluetooth headset with good sound quality and a comfortable design. It's priced around Rs 1299.

2. **realme Buds 2 Wired Headset**: If you're looking for a wired headset with boosted bass, the realme Buds 2 is a great option. It's priced around Rs 1299 and is considered one of the best earphones under this price range.

However, if you're willing to spend a bit more, you may want to consider other options like Sony or Bose, which are known for their high-quality sound. But if you're on a tight budget, the BoAt Airdopes 131 or realme Buds 2 are good options to consider.


In [36]:

response = run.invoke(
    {"input": "I like BoAt Airdopes"},
    config={"configurable": {"session_id": "user1"}}
)
print(response)

The BoAt Airdopes 131 Bluetooth Headset is a popular choice among music lovers. Here are some of its key features:

1. **Water-resistant design**: The BoAt Airdopes 131 has a water-resistant design, making it perfect for workouts or daily use.
2. **Long battery life**: The headset has a battery life of up to 5 hours on a single charge, and the charging case can provide an additional 15 hours of playback time.
3. **Type-C port**: The headset comes with a Type-C port for charging, making it convenient to charge your device.
4. **1-year warranty**: BoAt offers a 1-year warranty on the Airdopes 131, giving you peace of mind.
5. **Good sound quality**: The headset has good sound quality, with clear highs and decent bass.

However, some users have mentioned that the bass could be better, and the sound quality may not be as good as some other high-end headphones.

If you're interested in purchasing the BoAt Airdopes 131, I recommend checking out the latest prices and deals on online marketpla

In [37]:
response = run.invoke(
    {"input": "What about battery?"},
    config={"configurable": {"session_id": "user1"}}
)
print(response)

The battery life of the BoAt Airdopes 131 is one of its standout features. According to the reviews, the battery life is excellent. Here are some details:

* **Up to 5 hours of playback time**: The headset can provide up to 5 hours of playback time on a single charge.
* **70% battery remaining after 3 hours of use**: One user mentioned that after 3 hours of use, the battery was still at 70%, which is impressive.
* **Fast charging**: The charging case can provide an additional 15 hours of playback time, and it can charge the headset quickly.

Overall, the battery life of the BoAt Airdopes 131 is excellent, and it's a great option for those who want a headset that can last all day.

It's worth noting that the user also mentioned that the case is fully charged separately, which means you can charge the case and the headset at the same time, making it even more convenient.
